# Playground Series S6E9 - Will_Buy_EV

Binary classification, submission is a probability, metric ROC AUC.

Logic lives in `s6e9.py`; this notebook runs read -> train -> test and writes `submission.csv`.

On Kaggle: upload `s6e9.py` as a Dataset (or Utility Script) and add it as an input.

In [ ]:
import sys
from pathlib import Path
for p in Path("/kaggle/input").glob("*/s6e9.py") if Path("/kaggle/input").exists() else []:
    sys.path.append(str(p.parent))

import pandas as pd
import s6e9 as s

pd.set_option("display.width", 200)

## Read

In [ ]:
train, test = s.read_all()
print(train.shape, test.shape)
print(train[s.TARGET].mean())
train.head()

In [ ]:
# TODO EDA: distributions, target rate by category, train vs test drift
train.describe(include="all").T

In [ ]:
# permutation importance: quick fit on a sample, AUC drop when each feature is shuffled
from sklearn.inspection import permutation_importance

samp = train.sample(100_000, random_state=s.SEED)
Xs, _ = s.prep(samp)
ys = samp[s.TARGET]
cut = int(len(Xs) * 0.8)
m = s.make_model().fit(Xs.iloc[:cut], ys.iloc[:cut])
pi = permutation_importance(m, Xs.iloc[cut:], ys.iloc[cut:], scoring='roc_auc', n_repeats=5, random_state=s.SEED, n_jobs=-1)
pd.Series(pi.importances_mean, index=Xs.columns).sort_values(ascending=False).round(4)


## Train

In [ ]:
X, cats = s.prep(train)
y = train[s.TARGET]
X.dtypes

In [ ]:
# TODO feature engineering goes in s6e9.prep (keep notebook thin)
models, oof = s.train(X, y)

In [ ]:
# out-of-fold analysis: ROC curve, calibration by decile, AUC per category level
s.oof_report(X, y, oof)

## Test

In [ ]:
X_test, _ = s.prep(test, cats)
sub = s.test(models, X_test, test[s.ID])
sub.head()